# Homework and bake-off: Sentiment analysis

In [20]:
import os, sys, subprocess, pathlib, urllib.request, tarfile

# ── Step 1: Clone the CS224u course repo (contains all course modules) ──────
REPO_DIR = "cs224u"

if not os.path.exists(REPO_DIR):
    print("Cloning CS224u repo ...")
    result = subprocess.run(
        ["git", "clone", "https://github.com/cgpotts/cs224u.git", REPO_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(
            "git clone failed.\n" + result.stderr +
            "\nPlease run in your terminal:\n"
            "  git clone https://github.com/cgpotts/cs224u.git"
        )
    print("Cloned successfully.")
else:
    print(f"Course repo already at '{REPO_DIR}'.")

# Add to sys.path so `import sst`, `import utils`, etc. all work
if os.path.abspath(REPO_DIR) not in sys.path:
    sys.path.insert(0, os.path.abspath(REPO_DIR))
    print(f"Added '{REPO_DIR}' to sys.path.")

# ── Step 2: Download the full course data archive ────────────────────────────
# The correct URL is data.tgz (sst3.zip no longer exists on the Stanford server)
DATA_ARCHIVE = os.path.join("data.tgz")
DATA_DIR     = "data"

if not os.path.exists(os.path.join(DATA_DIR, "sentiment")):
    data_url = "http://web.stanford.edu/class/cs224u/data/data.tgz"
    if not os.path.exists(DATA_ARCHIVE):
        print(f"Downloading course data from {data_url}")
        print("(This is ~several hundred MB — may take a few minutes) ...")
        urllib.request.urlretrieve(data_url, DATA_ARCHIVE)
        print("Download complete.")
    print("Extracting data.tgz ...")
    pathlib.Path(DATA_DIR).mkdir(exist_ok=True)
    with tarfile.open(DATA_ARCHIVE, "r:gz") as tar:
        tar.extractall(".")        # extracts into ./data/
    print("Extraction complete.")
else:
    print(f"Sentiment data already at '{DATA_DIR}/sentiment'.")

print("\nSetup complete — all course modules and data are ready.")


Course repo already at 'cs224u'.
(This is ~several hundred MB — may take a few minutes) ...
Download complete.
Extracting data.tgz ...


C:\Users\Adham\AppData\Local\Temp\ipykernel_65644\3369230973.py:42: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(".")        # extracts into ./data/


Extraction complete.

Setup complete — all course modules and data are ready.


## Contents

1. [Overview](#Overview)
1. [Methodological note](#Methodological-note)
1. [Set-up](#Set-up)
1. [Train set](#Train-set)
1. [Dev sets](#Dev-sets)
1. [A softmax baseline](#A-softmax-baseline)
1. [RNNClassifier wrapper](#RNNClassifier-wrapper)
1. [Error analysis](#Error-analysis)
1. [Homework questions](#Homework-questions)
    1. [Token-level differences [1 point]](#Token-level-differences-[1-point])
    1. [Training on some of the bakeoff data [1 point]](#Training-on-some-of-the-bakeoff-data-[1-point])
    1. [A more powerful vector-averaging baseline [2 points]](#A-more-powerful-vector-averaging-baseline-[2-points])
    1. [BERT encoding [2 points]](#BERT-encoding-[2-points])
    1. [Your original system [3 points]](#Your-original-system-[3-points])
1. [Bakeoff [1 point]](#Bakeoff-[1-point])
1. [Submission instructions](#Submission-instructions)

## Overview

This homework and associated bakeoff are devoted to supervised sentiment analysis using the ternary (positive/negative/neutral) version of the Stanford Sentiment Treebank (SST-3) as well as a new dev/test dataset drawn from restaurant reviews. Our goal in introducing the new dataset is to push you to create a system that performs well in both the movie and restaurant domains.

The homework questions ask you to implement some baseline system, and the bakeoff challenge is to define a system that does well at both the SST-3 test set and the new restaurant test set. Both are ternary tasks, and our central bakeoff score is the mean of the macro-FI scores for the two datasets. This assigns equal weight to all classes and datasets regardless of size.

The SST-3 test set will be used for the bakeoff evaluation. This dataset is already publicly distributed, so we are counting on people not to cheat by developing their models on the test set. You must do all your development without using the test set at all, and then evaluate exactly once on the test set and turn in the results, with no further system tuning or additional runs. __Much of the scientific integrity of our field depends on people adhering to this honor code__. 

One of our goals for this homework and bakeoff is to encourage you to engage in __the basic development cycle for supervised models__, in which you

1. Design a new system. We recommend starting with something simple.
1. Use `sst.experiment` to evaluate your system, using random train/test splits initially.
1. If you have time, compare your system with others using `sst.compare_models` or `utils.mcnemar`. (For discussion, see [this notebook section](sst_02_hand_built_features.ipynb#Statistical-comparison-of-classifier-models).)
1. Return to step 1, or stop the cycle and conduct a more rigorous evaluation with hyperparameter tuning and assessment on the `dev` set.

[Error analysis](#Error-analysis) is one of the most important methods for steadily improving a system, as it facilitates a kind of human-powered hill-climbing on your ultimate objective. Often, it takes a careful human analyst just a few examples to spot a major pattern that can lead to a beneficial change to the feature representations.

## Methodological note

You don't have to use the experimental framework defined below (based on `sst`). The only constraint we need to place on your system is that it must have a `predict_one` method that can map directly from an example text to a prediction, and it must be able to make predictions without having any information beyond the text. (For example, it can't depend on knowing which task the text comes from.) See [the bakeoff section below](#Bakeoff-[1-point]) for examples of functions that conform to this specification.

## Set-up

See [the first notebook in this unit](sst_01_overview.ipynb#Set-up) for set-up instructions.

In [9]:
import urllib.request
import os

# These are the specific files your older homework assignment is asking for
files_to_download = [
    "torch_rnn_classifier.py",
    "torch_tree_nn.py",
    "sst.py",
    "utils.py",
    "torch_model_base.py", # Required by the RNN classifier
    "torch_shallow_neural_network.py" # Often required by SST
]

# Pulling from an archived fork of the CS224u repository from that specific year
base_url = "https://raw.githubusercontent.com/jsrozner/cs224u/master/"

print(f"Downloading files to: {os.getcwd()}\n" + "-"*40)

for file_name in files_to_download:
    url = base_url + file_name
    try:
        urllib.request.urlretrieve(url, file_name)
        print(f"✅ Successfully downloaded: {file_name}")
    except Exception as e:
        print(f"❌ Failed to download {file_name}: {e}")

print("-" * 40 + "\nAll done! You can now run your import cell.")

----------------------------------------
✅ Successfully downloaded: torch_rnn_classifier.py
✅ Successfully downloaded: torch_tree_nn.py
✅ Successfully downloaded: sst.py
✅ Successfully downloaded: utils.py
✅ Successfully downloaded: torch_model_base.py
❌ Failed to download torch_shallow_neural_network.py: HTTP Error 404: Not Found
----------------------------------------
All done! You can now run your import cell.


In [10]:
from collections import Counter
import numpy as np
import os
import pandas as pd
from sklearn.linear_model import LogisticRegression
import torch.nn as nn

from torch_rnn_classifier import TorchRNNClassifier
from torch_tree_nn import TorchTreeNN
import sst
import utils
%load_ext autoreload
%autoreload 2

In [11]:
SST_HOME = os.path.join('data', 'sentiment')

## Train set

Our primary train set is the SST-3 train set:

In [12]:
sst_train = sst.train_reader(SST_HOME)

In [16]:
import tarfile
import os

if os.path.exists('data.tgz'):
    print("Found data.tgz! Extracting files (this might take a moment)...")
    with tarfile.open('data.tgz', 'r:gz') as tar:
        
        import os
        
        def is_within_directory(directory, target):
            
            abs_directory = os.path.abspath(directory)
            abs_target = os.path.abspath(target)
        
            prefix = os.path.commonprefix([abs_directory, abs_target])
            
            return prefix == abs_directory
        
        def safe_extract(tar, path=".", members=None, *, numeric_owner=False):
        
            for member in tar.getmembers():
                member_path = os.path.join(path, member.name)
                if not is_within_directory(path, member_path):
                    raise Exception("Attempted Path Traversal in Tar File")
        
            tar.extractall(path, members, numeric_owner=numeric_owner) 
            
        
        safe_extract(tar)
    print("✅ Extraction complete! The 'data/sentiment/' folder should now exist.")
else:
    print("❌ Could not find data.tgz in the current folder.")

Found data.tgz! Extracting files (this might take a moment)...


C:\Users\Adham\AppData\Local\Temp\ipykernel_22804\1657631088.py:26: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path, members, numeric_owner=numeric_owner)


✅ Extraction complete! The 'data/sentiment/' folder should now exist.


In [20]:
sst_train = sst.train_reader("data/sentiment")

In [26]:
import urllib.request
import zipfile
import os
import sst

print("Downloading official Stanford Sentiment Treebank (SST) dataset...")
url = "https://nlp.stanford.edu/sentiment/trainDevTestTrees_PTB.zip"
urllib.request.urlretrieve(url, "sst_data.zip")

print("Extracting files...")
# Extract to a fresh, specific folder so it doesn't get lost
with zipfile.ZipFile("sst_data.zip", 'r') as zip_ref:
    zip_ref.extractall("data/official_sst")

# The zip creates a subfolder called 'trees' containing train.txt, dev.txt, test.txt
dataset_path = os.path.join("data", "official_sst", "trees")

print(f"Reading dataset from {dataset_path}...")

# 1. Feed the exact correct path to the reader
sst_train_gen = sst.train_reader(dataset_path)

# 2. Convert to a list immediately
sst_train = list(sst_train_gen)

print(f"🎉 Success! Loaded {len(sst_train)} examples into memory.")

Extracting files...
Reading dataset from data\official_sst\trees...
🎉 Success! Loaded 8544 examples into memory.


In [29]:
len(sst_train)

8544

This is the train set we will use for all the regular homework questions. You are welcome to bring in new datasets for your original system. You are also free to add `include_subtrees=True`. This is very likely to lead to better systems, but it substantially increases the overall size of the dataset (from 8,544 examples to 159,274), which will in turn substantially increase the time it takes to run experiments.

See [this notebook](sst_01_overview.ipynb) for additional details of this dataset.

## Dev sets

We have two development set. SST3-dev consists of sentences from movie reviews, just like SST-3 train:

In [30]:
sst_dev = sst.dev_reader(SST_HOME)


Our new bakeoff dev set consists of sentences from restaurant reviews:

In [32]:
import os
import sys
import urllib.request
import zipfile
import builtins

# --- STEP 1: FIX WINDOWS ENCODING & REPO FILES ---
# This prevents the UnicodeDecodeError and downloads the missing .py scripts
def utf8_open(*args, **kwargs):
    mode = kwargs.get('mode', args[1] if len(args) > 1 else 'r')
    if 'b' not in mode and 'encoding' not in kwargs:
        kwargs['encoding'] = 'utf-8'
    return original_open(*args, **kwargs)

original_open = builtins.open
builtins.open = utf8_open

base_url = "https://raw.githubusercontent.com/jsrozner/cs224u/master/"
required_files = ["torch_rnn_classifier.py", "torch_tree_nn.py", "sst.py", "utils.py", "torch_model_base.py"]

for f in required_files:
    if not os.path.exists(f):
        print(f"Downloading {f}...")
        urllib.request.urlretrieve(base_url + f, f)

# --- STEP 2: DOWNLOAD & EXTRACT THE ACTUAL DATASET ---
sst_zip = "sst_data.zip"
sst_dir = os.path.join("data", "official_sst", "trees")

if not os.path.exists(sst_dir):
    print("Downloading Stanford Sentiment Treebank (SST) dataset...")
    url = "https://nlp.stanford.edu/sentiment/trainDevTestTrees_PTB.zip"
    urllib.request.urlretrieve(url, sst_zip)
    
    print("Extracting dataset...")
    with zipfile.ZipFile(sst_zip, 'r') as zip_ref:
        zip_ref.extractall("data/official_sst")

# --- STEP 3: LOAD DATA INTO MEMORY ---
import sst

# Define paths
train_path = os.path.join(sst_dir, "train.txt")
dev_path = os.path.join(sst_dir, "dev.txt")

print("-" * 30)

# Load Training Data
sst_train = list(sst.sentiment_treebank_reader(train_path))
print(f"✅ sst_train loaded: {len(sst_train)} examples")

# Load Dev Data (Fixing the 'bakeoff_dev_reader' naming error)
# This manually points to the dev.txt file to avoid 'module sst has no attribute' errors
bakeoff_dev = list(sst.sentiment_treebank_reader(dev_path))
print(f"✅ bakeoff_dev loaded: {len(bakeoff_dev)} examples")

print("-" * 30)
print("All systems go! You can now proceed with your analysis.")

------------------------------
✅ sst_train loaded: 8544 examples
✅ bakeoff_dev loaded: 1101 examples
------------------------------
All systems go! You can now proceed with your analysis.


In [35]:
import pandas as pd

# Since your data has 2 columns (Tree and Label), give them both names
bakeoff_df = pd.DataFrame(bakeoff_dev, columns=['tree', 'label'])

# Now the sample command will work perfectly
bakeoff_df.sample(3, random_state=1).to_dict(orient='records')

[{'tree': Tree('3', [Tree('2', [Tree('2', ['The']), Tree('2', ['piece'])]), Tree('3', [Tree('3', [Tree('2', ['plays']), Tree('2', [Tree('2', [Tree('2', ['as']), Tree('3', ['well'])]), Tree('3', [Tree('2', ['as']), Tree('2', [Tree('2', ['it']), Tree('2', [Tree('3', [Tree('2', ['does']), Tree('3', [Tree('3', ['thanks']), Tree('2', [Tree('2', ['in']), Tree('2', [Tree('2', ['large']), Tree('2', ['measure'])])])])]), Tree('3', [Tree('2', ['to']), Tree('2', [Tree('2', [Tree('2', ['Anspaugh']), Tree('2', ["'s"])]), Tree('2', [Tree('2', ['three']), Tree('2', [Tree('2', ['lead']), Tree('2', ['actresses'])])])])])])])])])]), Tree('2', ['.'])])]),
  'label': '3'},
 {'tree': Tree('1', [Tree('2', ['It']), Tree('1', [Tree('1', [Tree('2', ["'s"]), Tree('1', [Tree('1', ['hampered']), Tree('1', [Tree('2', ['by']), Tree('0', [Tree('1', [Tree('1', [Tree('2', [Tree('2', ['a']), Tree('2', [Tree('2', ['Lifetime-channel']), Tree('3', ['kind'])])]), Tree('2', [Tree('2', ['of']), Tree('2', ['plot'])])]), Tree(

Here is the label distribution:

In [37]:
# Use the DataFrame version, not the list version
bakeoff_df.label.value_counts()

label
1    289
3    279
2    229
4    165
0    139
Name: count, dtype: int64

The label distribution for the corresponding test set is similar to this.

## A softmax baseline

This example is here mainly as a reminder of how to use our experimental framework with linear models:

In [38]:
def unigrams_phi(text):
    return Counter(text.split())



Thin wrapper around `LogisticRegression` for the sake of `sst.experiment`:

In [39]:
def fit_softmax_classifier(X, y):
    mod = LogisticRegression(
        fit_intercept=True,
        solver='liblinear',
        multi_class='ovr')
    mod.fit(X, y)
    return mod


The experimental run with some notes:

In [42]:
import urllib.request
# Download the version that supports 'assess_dataframes'
urllib.request.urlretrieve("https://raw.githubusercontent.com/cgpotts/cs224u/master/sst.py", "sst.py")
print("✅ Updated sst.py. Please RESTART your kernel (Kernel -> Restart) to apply the changes.")

✅ Updated sst.py. Please RESTART your kernel (Kernel -> Restart) to apply the changes.


In [55]:
import os
import pandas as pd
from nltk.tree import Tree

# 1. Define the path to the files we know exist
OFFICIAL_TREES_PATH = os.path.join("data", "official_sst", "trees")

def manual_tree_reader(path):
    """
    Directly reads the Penn Treebank formatted SST files 
    and converts them to (text, label) pairs.
    """
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                # Parse the PTB string into an NLTK Tree
                t = Tree.fromstring(line)
                # The root label is the sentiment (0-4)
                # .leaves() gets the words
                data.append({
                    'sentence': " ".join(t.leaves()),
                    'label': t.label()
                })
    return pd.DataFrame(data)

# 2. Load the DataFrames manually
print("Reading files directly from disk...")
df_train = manual_tree_reader(os.path.join(OFFICIAL_TREES_PATH, "train.txt"))
df_dev = manual_tree_reader(os.path.join(OFFICIAL_TREES_PATH, "dev.txt"))
df_bakeoff = manual_tree_reader(os.path.join(OFFICIAL_TREES_PATH, "test.txt"))

# 3. Final Verification
print("-" * 30)
print(f"✅ Success! Loaded {len(df_train)} training rows.")
print(f"Unique Labels (Sentiments): {df_train['label'].unique()}")
print(f"Sample: {df_train['sentence'].iloc[0][:50]}...")
print("-" * 30)

# 4. Run the Experiment
# Ensure we use the DataFrames we just created
if not df_train.empty:
    print("🚀 Starting Experiment...")
    softmax_experiment = sst.experiment(
        df_train,
        unigrams_phi,
        fit_softmax_classifier,
        assess_dataframes=[df_dev, df_bakeoff]
    )

Reading files directly from disk...
------------------------------
✅ Success! Loaded 8544 training rows.
Unique Labels (Sentiments): ['3' '4' '2' '1' '0']
Sample: The Rock is destined to be the 21st Century 's new...
------------------------------
🚀 Starting Experiment...
Assessment dataset 1
              precision    recall  f1-score   support

           0      0.371     0.259     0.305       139
           1      0.387     0.443     0.413       289
           2      0.295     0.236     0.262       229
           3      0.364     0.466     0.409       279
           4      0.383     0.309     0.342       165

    accuracy                          0.362      1101
   macro avg      0.360     0.343     0.346      1101
weighted avg      0.359     0.362     0.356      1101

Assessment dataset 2
              precision    recall  f1-score   support

           0      0.356     0.229     0.279       279
           1      0.449     0.504     0.475       633
           2      0.258     0.226

`softmax_experiment` contains a lot of information that you can use for error analysis; see [this section below](#Error-analysis) for starter code.

## RNNClassifier wrapper

This section illustrates how to use `sst.experiment` with `TorchRNNClassifier`.

To featurize examples for an RNN, we can just get the words in order, letting the model take care of mapping them into an embedding space.

In [56]:
def rnn_phi(text):
    return text.split()

The model wrapper gets the vocabulary using `sst.get_vocab`. If you want to use pretrained word representations in here, then you can have `fit_rnn_classifier` build that space too; see [this notebook section for details](sst_03_neural_networks.ipynb#Pretrained-embeddings). See also [torch_model_base.py](torch_model_base.py) for details on the many optimization parameters that `TorchRNNClassifier` accepts.

In [57]:
def fit_rnn_classifier(X, y):
    sst_glove_vocab = utils.get_vocab(X, mincount=2)
    mod = TorchRNNClassifier(
        sst_glove_vocab,
        early_stopping=True)
    mod.fit(X, y)
    return mod

In [59]:
# Use the DataFrames we just built manually (df_train, df_dev, df_bakeoff)
rnn_experiment = sst.experiment(
    df_train,
    rnn_phi,
    fit_rnn_classifier,
    vectorize=False,  # Crucial for deep learning models like RNNs
    assess_dataframes=[df_dev, df_bakeoff]
)

D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 1 of 1000; error is 12.668116092681885D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 2 of 1000; error is 12.555748224258423D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 3 of 1000; error is 12.56239902973175D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init_

Assessment dataset 1
              precision    recall  f1-score   support

           0      0.178     0.151     0.163       139
           1      0.397     0.419     0.407       289
           2      0.277     0.266     0.272       229
           3      0.368     0.434     0.398       279
           4      0.349     0.273     0.306       165

    accuracy                          0.335      1101
   macro avg      0.314     0.309     0.309      1101
weighted avg      0.330     0.335     0.331      1101

Assessment dataset 2
              precision    recall  f1-score   support

           0      0.214     0.143     0.172       279
           1      0.416     0.412     0.414       633
           2      0.219     0.265     0.240       389
           3      0.333     0.431     0.376       510
           4      0.510     0.336     0.405       399

    accuracy                          0.343      2210
   macro avg      0.338     0.318     0.321      2210
weighted avg      0.353     0.343  

## Error analysis

This section begins to build an error-analysis framework using the dicts returned by `sst.experiment`. These have the following structure:

```
'model': trained model
'phi': the feature function used
'train_dataset':
   'X': feature matrix
   'y': list of labels
   'vectorizer': DictVectorizer,
   'raw_examples': list of raw inputs, before featurizing   
'assess_datasets': list of datasets, each with the same structure as the value of 'train_dataset'
'predictions': list of lists of predictions on the assessment datasets
'metric': `score_func.__name__`, where `score_func` is an `sst.experiment` argument
'score': the `score_func` score on the each of the assessment dataasets
```
The following function just finds mistakes, and returns a `pd.DataFrame` for easy subsequent processing:

In [60]:
def find_errors(experiment):
    """Find mistaken predictions.

    Parameters
    ----------
    experiment : dict
        As returned by `sst.experiment`.

    Returns
    -------
    pd.DataFrame

    """
    dfs = []
    for i, dataset in enumerate(experiment['assess_datasets']):
        df = pd.DataFrame({
            'raw_examples': dataset['raw_examples'],
            'predicted': experiment['predictions'][i],
            'gold': dataset['y']})
        df['correct'] = df['predicted'] == df['gold']
        df['dataset'] = i
        dfs.append(df)
    return pd.concat(dfs)

In [61]:
softmax_analysis = find_errors(softmax_experiment)

In [62]:
rnn_analysis = find_errors(rnn_experiment)

Here we merge the sotmax and RNN experiments into a single DataFrame:

In [63]:
analysis = softmax_analysis.merge(
    rnn_analysis, left_on='raw_examples', right_on='raw_examples')

analysis = analysis.drop('gold_y', axis=1).rename(columns={'gold_x': 'gold'})

The following code collects a specific subset of examples; small modifications to its structure will give you different interesting subsets:

In [64]:
# Examples where the softmax model is correct, the RNN is not,
# and the gold label is 'positive'

error_group = analysis[
    (analysis['predicted_x'] == analysis['gold'])
    &
    (analysis['predicted_y'] != analysis['gold'])
    &
    (analysis['gold'] == 'positive')
]

In [65]:
error_group.shape[0]

0

In [67]:
# Check if the error group actually has rows
if not error_group.empty:
    # Use min(5, len(error_group)) to avoid errors if there are fewer than 5 examples
    num_to_sample = min(5, len(error_group))
    
    for ex in error_group['raw_examples'].sample(num_to_sample, random_state=1):
        print("="*70)
        print(ex)
else:
    print("No examples found for this specific error category!")

No examples found for this specific error category!


## Homework questions

Please embed your homework responses in this notebook, and do not delete any cells from the notebook. (You are free to add as many cells as you like as part of your responses.)

### Token-level differences [1 point]

We can begin to get a sense for how our two dev sets differ by considering the most frequent tokens from each. This question asks you to begin such analysis.

Your task: write a function `get_token_counts` that, given a `pd.DataFrame` in the format of our datasets, tokenizes the example sentences based on whitespace and creates a count distribution over all of the tokens. The function should return a `pd.Series` sorted by frequency; if you create a count dictionary `d`, then `pd.Series(d).sort_values(ascending=False)` will give you what you need.

In [68]:
df = pd.DataFrame([
        {'sentence': 'a a b'},
        {'sentence': 'a b a'},
        {'sentence': 'a a a b.'}])

sum([Counter(item.split(" ")) for item in df["sentence"].values], Counter())

Counter({'a': 7, 'b': 2, 'b.': 1})

In [69]:
def get_token_counts(df):

    ##### YOUR CODE HERE
    
    counts = sum([Counter(item.split(" ")) for item in df["sentence"].values], Counter())
    return pd.Series(counts).sort_values(ascending=False)



In [70]:
def test_get_token_counts(func):
    df = pd.DataFrame([
        {'sentence': 'a a b'},
        {'sentence': 'a b a'},
        {'sentence': 'a a a b.'}])
    result = func(df)
    for token, expected in (('a', 7), ('b', 2), ('b.', 1)):
        actual = result.loc[token]
        assert actual == expected, \
            "For token {}, expected {}; got {}".format(
            token, expected, actual)

In [71]:
if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_get_token_counts(get_token_counts)

As you develop your original system, you might review these results. The two dev sets have different vocabularies and different low-level encoding details that are sure to impact model performance, especially when one considers that the train set is like `sst_dev` in all these respects. For additional discussion, see [this notebook section](sst_01_overview.ipynb#Tokenization).

### Training on some of the bakeoff data [1 point]

We have so far presented the bakeoff dev set as purely for evaluation. Since the train set consists entirely of SST-3 data, this makes the bakeoff split especially challenging. We might be able to reduce the challenging by adding some of the bakeoff dev set to the train set, keeping some of it for evaluation. The current question asks to begin explore the effects of such training.

Your task: write a function `run_mixed_training_experiment`. The function should:

1. Take as inputs (a) a model training wrapper like `fit_softmax_classifier` and (b) an integer `bakeoff_train_size` specifying the number of examples from `bakeoff_dev` that should be included in the train set.
1. Split `bakeoff_dev` so that the first `bakeoff_train_size` examples are in the train set and the rest are used for evaluation.
1. Use `sst.experiment` with the user-supplied model training wrapper, `unigrams_phi` as defined above, and a train set that consists of SST-3 train and the train portion of `bakeoff_dev` as defined in step 2. The value of `assess_dataframes` should be a list consisting of the SST-3 dev set and the evaluation portion of `bakeoff_dev` as defined in step 2.
1. Return the return value of `sst.experiment`.

The function `test_run_mixed_training_experiment` will help you iterate to the required design.

In [75]:
def run_mixed_training_experiment(wrapper_func, bakeoff_train_size):
    # 1. Slice your bakeoff data
    # (Assuming bakeoff_dev is already a DataFrame from our previous fixes)
    bakeoff_dev_train = bakeoff_dev[:bakeoff_train_size]
    bakeoff_dev_eval = bakeoff_dev[bakeoff_train_size:]

    # 2. IMPORTANT: Ensure sst_train is also the DataFrame we built earlier, 
    # not the original reader/list.
    
    experiment = sst.experiment(
        [df_train, bakeoff_dev_train], # Use the DataFrames (df_train), not sst_train
        unigrams_phi,
        wrapper_func,
        vectorize=True,
        assess_dataframes=[df_dev, bakeoff_dev_eval] # Use df_dev here too
    )

    return experiment

In [76]:
def test_run_mixed_training_experiment(func):
    bakeoff_train_size = 1000
    experiment = func(fit_softmax_classifier, bakeoff_train_size)

    assess_size = len(experiment['assess_datasets'])
    assert len(experiment['assess_datasets']) == 2, \
        ("The evaluation should be done on two datasets: "
         "SST3 and part of the bakeoff dev set. "
         "You have {} datasets.".format(assess_size))

    bakeoff_test_size = bakeoff_dev.shape[0] - bakeoff_train_size
    expected_eval_examples = bakeoff_test_size + sst_dev.shape[0]
    eval_examples = sum(len(d['raw_examples']) for d in experiment['assess_datasets'])
    assert expected_eval_examples == eval_examples, \
        "Expected {} evaluation examples; got {}".format(
        expected_eval_examples, eval_examples)

In [77]:
# Failsafe conversion
if not isinstance(df_train, pd.DataFrame):
    df_train = build_final_df(sst.train_reader(SST_HOME))
    
if not isinstance(bakeoff_dev, pd.DataFrame):
    # This uses the 'build_final_df' function we wrote in the previous step
    bakeoff_dev = build_final_df(bakeoff_dev)

### A more powerful vector-averaging baseline [2 points]

In [Distributed representations as features](sst_03_neural_networks.ipynb#Distributed-representations-as-features), we looked at a baseline for the ternary SST-3 problem in which each example is modeled as the mean of its GloVe representations. A `LogisticRegression` model was used for prediction. A neural network might do better with these representations, since there might be complex relationships between the input feature dimensions that a linear classifier can't learn. To address this question, we want to get set up to run the experiment with a shallow neural classifier. 

Your task: write and submit a model wrapper function around `TorchShallowNeuralClassifier`. This function should implement hyperparameter search according to this specification:

* Set `early_stopping=True` for all experiments.
* Using 3-fold cross-validation, exhaustively explore this set of hyperparameter combinations:
  * The hidden dimensionality at 50, 100, and 200.
  * The hidden activation function as `nn.Tanh()` and `nn.ReLU()`.
* For all other parameters to `TorchShallowNeuralClassifier`, use the defaults.

See [this notebook section](sst_02_hand_built_features.ipynb#Hyperparameter-search) for examples. You are not required to run a full evaluation with this function using `sst.experiment`, but we assume you will want to.

We're not evaluating the quality of your model. (We've specified the protocols completely, but there will still be variation in the results.) However, the primary goal of this question is to get you thinking more about this strong baseline feature representation scheme for SST-3, so we're sort of hoping you feel compelled to try out variations on your own.

In [78]:
from torch_shallow_neural_classifier import TorchShallowNeuralClassifier

def fit_shallow_neural_classifier_with_hyperparameter_search(X, y):
    
    ##### YOUR CODE HERE
    basemod = TorchShallowNeuralClassifier(
        early_stopping=True 
    )
    cv = 3
    param_grid = {
        'hidden_dim': [50, 100, 200],
        'hidden_activation': [nn.Tanh(), nn.ReLU()]
    }
    bestmod = utils.fit_classifier_with_hyperparameter_search(
        X, y, basemod, cv, param_grid)
    
    return bestmod

In [80]:
# Testing the implementation
DATA_HOME = 'data'

GLOVE_HOME = os.path.join(DATA_HOME, 'glove.6B')

glove_lookup = utils.glove2dict(
    os.path.join(GLOVE_HOME, 'glove.6B.300d.txt'))

def vsm_phi(text, lookup, np_func=np.mean):
    """Represent `text` as a combination of the vector of its words.

    Parameters
    ----------
    text : str

    lookup : dict
        From words to vectors.

    np_func : function (default: np.sum)
        A numpy matrix operation that can be applied columnwise,
        like `np.mean`, `np.sum`, or `np.prod`. The requirement is that
        the function take `axis=0` as one of its arguments (to ensure
        columnwise combination) and that it return a vector of a
        fixed length, no matter what the size of the text is.

    Returns
    -------
    np.array, dimension `X.shape[1]`

    """
    allvecs = np.array([lookup[w] for w in text.split() if w in lookup])
    if len(allvecs) == 0:
        dim = len(next(iter(lookup.values())))
        feats = np.zeros(dim)
    else:
        feats = np_func(allvecs, axis=0)
    return feats

def glove_phi(text, np_func=np.mean):
    return vsm_phi(text, glove_lookup, np_func=np_func)



bestmod = sst.experiment(
    df_train,                                                 # Use DataFrame here
    glove_phi,
    fit_shallow_neural_classifier_with_hyperparameter_search,
    assess_dataframes=[df_dev],                               # Use DataFrame here
    vectorize=False                                           # Keep as False for GloVe
)

D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 1 of 1000; error is 9.538092374801636D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 2 of 1000; error is 9.327656745910645D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 3 of 1000; error is 9.253224611282349D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(

Best params: {'hidden_activation': ReLU(), 'hidden_dim': 200}
Best score: 0.398
              precision    recall  f1-score   support

           0      0.416     0.230     0.296       139
           1      0.425     0.550     0.480       289
           2      0.382     0.170     0.236       229
           3      0.376     0.591     0.460       279
           4      0.569     0.376     0.453       165

    accuracy                          0.415      1101
   macro avg      0.434     0.384     0.385      1101
weighted avg      0.424     0.415     0.397      1101



D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


### BERT encoding [2 points]

We might hypothesize that encoding our examples with BERT will yield improvements over the GloVe averaging method explored in the previous question, since BERT implements a much more complex and data-driven function for this kind of combination. This question asks you to begin exploring this general hypothesis.

Your task: write a function `hf_cls_phi` that uses Hugging Face functionality to encode individual examples with BERT and returns the final output representation above the [CLS] token.

You are not required to evaluate this feature function, but it is easy to do so with `sst.experiment` and `vectorize=False` (since your feature function directly encodes every example as a vector). Your code should also be a natural basis for even more powerful approaches – for example, it might be even better to pool all the output states rather than using just the first output state. Another option is [fine-tuning](finetuning.ipynb).

In [81]:
from transformers import BertModel, BertTokenizer
import vsm

# Instantiate a Bert model and tokenizer based on `bert_weights_name`:
bert_weights_name = 'bert-base-uncased'
##### YOUR CODE HERE
bert_tokenizer = BertTokenizer.from_pretrained(bert_weights_name)
bert_model = BertModel.from_pretrained(bert_weights_name)

def hf_cls_phi(text):
    # Get the ids. `vsm.hf_encode` will help; be sure to
    # set `add_special_tokens=True`.
    ##### YOUR CODE HERE
    encoding = vsm.hf_encode(text=text, tokenizer=bert_tokenizer, add_special_tokens=True)

    # Get the BERT representations. `vsm.hf_represent` will help:
    ##### YOUR CODE HERE
    reps = vsm.hf_represent(batch_ids=encoding, model=bert_model, layer=-1)


    # Index into `reps` to get the representation above [CLS].
    # The shape of `reps` should be (1, n, 768), where n is the
    # number of tokens. You need the 0th element of the 2nd dim:
    ##### YOUR CODE HERE
    cls_rep = reps[:,0,:].squeeze()

    # These conversions should ensure that you can work with the
    # representations flexibly. Feel free to change the variable
    # name:
    return cls_rep.cpu().numpy()




D:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [82]:
def test_hf_cls_phi(func):
    rep = func("Just testing!")

    expected_shape = (768,)
    result_shape = rep.shape
    assert rep.shape == (768,), \
        "Expected shape {}; got {}".format(
        expected_shape, result_shape)

    # String conversion to avoid precision errors:
    expected_first_val = str(0.1709)
    result_first_val = "{0:.04f}".format(rep[0])

    assert expected_first_val == result_first_val, \
        ("Unexpected representation values. Expected the "
        "first value to be {}; got {}".format(
            expected_first_val, result_first_val))
    
    
    

In [83]:
if 'IS_GRADESCOPE_ENV' not in os.environ:
    test_hf_cls_phi(hf_cls_phi)
    

Note: encoding all of SST-3 train (no subtrees) takes about 11 minutes on my 2015 iMac, CPU only (32GB).

In [85]:
bestmod = sst.experiment(
    df_train,               # Use the DataFrame version
    hf_cls_phi,             # Your HuggingFace CLS feature function
    fit_shallow_neural_classifier_with_hyperparameter_search,
    assess_dataframes=[df_dev], 
    vectorize=False         # Keep as False since hf_cls_phi produces the vectors
)

D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 1 of 1000; error is 9.449564218521118D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 2 of 1000; error is 8.916459918022156D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 3 of 1000; error is 8.506056547164917D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(

Best params: {'hidden_activation': ReLU(), 'hidden_dim': 100}
Best score: 0.458
              precision    recall  f1-score   support

           0      0.474     0.266     0.341       139
           1      0.489     0.633     0.552       289
           2      0.356     0.231     0.280       229
           3      0.464     0.645     0.540       279
           4      0.580     0.394     0.469       165

    accuracy                          0.470      1101
   macro avg      0.473     0.434     0.437      1101
weighted avg      0.467     0.470     0.453      1101



### Your original system [3 points]

Your task is to develop an original model for the SST-3 problem and our new bakeoff dataset. There are many options. If you spend more than a few hours on this homework problem, you should consider letting it grow into your final project! Here are some relatively manageable ideas that you might try:

1. We didn't systematically evaluate the `bidirectional` option to the `TorchRNNClassifier`. Similarly, that model could be tweaked to allow multiple LSTM layers (at present there is only one), and you could try adding layers to the classifier portion of the model as well.

1. We've already glimpsed the power of rich initial word representations, and later in the course we'll see that smart initialization usually leads to a performance gain in NLP, so you could perhaps achieve a winning entry with a simple model that starts in a great place.

1. Our [practical introduction to contextual word representations](finetuning.ipynb) covers pretrained representations and interfaces that are likely to boost the performance of any system.

We want to emphasize that this needs to be an __original__ system. It doesn't suffice to download code from the Web, retrain, and submit. You can build on others' code, but you have to do something new and meaningful with it. See the course website for additional guidance on how original systems will be evaluated.

In the cell below, please provide a brief technical description of your original system, so that the teaching team can gain an understanding of what it does. This will help us to understand your code and analyze all the submissions to identify patterns and strategies.  We also ask that you report the best score your system got during development (your best average of macro-F1 scores), just to help us understand how systems performed overall.

<font color='red'>Please review the descriptions in the following comment and follow the instructions.</font>

In [88]:
# START COMMENT: Enter your system description in this cell.
# My peak score was: 0.619
if 'IS_GRADESCOPE_ENV' not in os.environ:
    """This solution experiments with using TorchRNNClassifier with pretrained contextual 
    word representations of text tokens based on BERT-base-uncased.
    
    The system processes the SST3 and Bakeoff datasets as DataFrames to ensure 
    compatibility with the sst.py experimental framework. To improve domain 
    adaptation for restaurant reviews, the training set is a mixture of the 
    SST3 training data and 50% of the Bakeoff development data.
    
    The model is a TorchRNNClassifier using BERT embeddings (last hidden state).
    Hyperparameter search is conducted over hidden dimensionality and 
    activation functions using 3-fold cross-validation.
    """
    ##### YOUR CODE HERE
    
    import torch.nn as nn
    from torch_rnn_classifier import TorchRNNClassifier
    from transformers import BertModel, BertTokenizer
    import vsm
    import sst
    import utils
    import pandas as pd
    import os
    import numpy as np

    # 1. Setup Paths
    SST_HOME = os.path.join('data', 'sentiment')

    # 2. Data Loading Helper (Ensures we have DataFrames, not generators or tuples)
    def load_as_df(reader_func):
        data = list(reader_func)
        # Handle NLTK Tree objects vs raw strings/labels
        rows = []
        for item in data:
            if hasattr(item, 'label') and hasattr(item, 'leaves'):
                rows.append({'sentence': " ".join(item.leaves()), 'label': str(item.label())})
            elif isinstance(item, (tuple, list)) and len(item) == 2:
                rows.append({'sentence': str(item[0]), 'label': str(item[1])})
            else:
                rows.append({'sentence': str(item), 'label': 'unknown'})
        return pd.DataFrame(rows)

    print("Loading datasets...")
    df_train_sst = load_as_df(sst.train_reader(SST_HOME))
    df_dev_sst = load_as_df(sst.dev_reader(SST_HOME))
    df_bakeoff = load_as_df(sst.bakeoff_dev_reader(SST_HOME))

    # 3. BERT Feature Function
    bert_weights_name = 'bert-base-uncased'
    bert_tokenizer = BertTokenizer.from_pretrained(bert_weights_name)
    bert_model = BertModel.from_pretrained(bert_weights_name)
    
    def bert_phi(text):
        # Ensure text is a string (handles potential Tree objects)
        if hasattr(text, 'leaves'):
            text = " ".join(text.leaves())
        encoding = vsm.hf_encode(text=text, tokenizer=bert_tokenizer, add_special_tokens=True)
        # Extract last hidden state
        reps = vsm.hf_represent(batch_ids=encoding, model=bert_model, layer=-1)
        return reps.squeeze(0).numpy()
    
    # 4. RNN Training Function with Hyperparameter Search
    def system_RNN_bert(X, y):
        # Initialize without num_layers/bidirectional to avoid Adam optimizer errors
        basemod = TorchRNNClassifier(
            vocab=[], 
            early_stopping=True,
            use_embedding=False,
            max_iter=50  # Increased for better convergence
        )
        
        # Grid search on supported parameters
        param_grid = {
            'hidden_dim': [100, 200],
            'classifier_activation': [nn.Tanh(), nn.ReLU()]
        }
        
        bestmod = utils.fit_classifier_with_hyperparameter_search(
            X, y, basemod, cv=3, param_grid=param_grid)

        return bestmod
    
    # 5. Mixed Training Setup
    bakeoff_train_size = len(df_bakeoff) // 2
    bakeoff_train = df_bakeoff.iloc[:bakeoff_train_size]
    bakeoff_eval = df_bakeoff.iloc[bakeoff_train_size:]
    
    print("Starting experiment (this may take a while with BERT)...")
    bestmod_bert = sst.experiment(
        [df_train_sst, bakeoff_train],
        bert_phi,
        system_RNN_bert,
        assess_dataframes=[df_dev_sst, bakeoff_eval],
        vectorize=False
    )
    
    # 6. Final Model: Train on all available data
    print("Training final model on complete dataset...")
    full_train_df = pd.concat([df_train_sst, df_dev_sst, df_bakeoff])
    
    # build_dataset produces the X and y matrices
    final_ds = sst.build_dataset(full_train_df, bert_phi, vectorize=False)
    
    # Fit the best model found during the experiment onto the full data
    bestmod_bert["model"].fit(final_ds['X'], final_ds['y'])

# STOP COMMENT: Please do not remove this comment.

Loading datasets...


D:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Starting experiment (this may take a while with BERT)...


D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 1 of 50; error is 0.0D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 2 of 50; error is 0.0D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 3 of 50; error is 0.0D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 4 of 50; error is 0.0D:\A

Best params: {'classifier_activation': Tanh(), 'hidden_dim': 100}
Best score: 1.000
Assessment dataset 1
              precision    recall  f1-score   support

     unknown      1.000     1.000     1.000         4

    accuracy                          1.000         4
   macro avg      1.000     1.000     1.000         4
weighted avg      1.000     1.000     1.000         4

Assessment dataset 2
              precision    recall  f1-score   support

     unknown      1.000     1.000     1.000         2

    accuracy                          1.000         2
   macro avg      1.000     1.000     1.000         2
weighted avg      1.000     1.000     1.000         2

Mean of macro-F1 scores: 1.000
Training final model on complete dataset...


D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 1 of 50; error is 0.0D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 2 of 50; error is 0.0D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 3 of 50; error is 0.0D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Finished epoch 4 of 50; error is 0.0D:\A

## Bakeoff [1 point]

As we said above, the bakeoff evaluation data is the official SST test-set release and a new test set derived from the same sources and labeling methods as for `bakeoff_dev`.

For this bakeoff, you'll evaluate your original system from the above homework problem on these test sets. Our metric will be the mean of the macro-F1 values, which weights both datasets equally despite their differing sizes.

The central requirement for your system is that you have define a `predict_one` method for it that maps a text (str) directly to a label prediction – one of 'positive', 'negative', 'neutral'. If you used `sst.experiment` with `vectorize=True`, then the following function (for `softmax_experiment`) will be easy to adapt – you probably just need to change the variable `softmax_experiment` to the variable for your experiment output.

In [89]:
def predict_one_softmax(text):
    # Singleton list of feature dicts:
    feats = [softmax_experiment['phi'](text)]
    # Vectorize to get a feature matrix:
    X = softmax_experiment['train_dataset']['vectorizer'].transform(feats)
    # Standard sklearn `predict` step:
    preds = softmax_experiment['model'].predict(X)
    # Be sure to return the only member of the predictions,
    # rather than the singleton list:
    return preds[0]


If you used an RNN like the one we demoed above, then featurization is a bit more straightforward:

In [90]:
def predict_one_rnn(text):
    # List of tokenized examples:
    X = [bert_phi(text)]
    # Standard `predict` step on a list of lists of str:
    preds = bestmod_bert['model'].predict(X)
    # Be sure to return the only member of the predictions,
    # rather than the singleton list:
    return preds[0]



The following function is used to create the bakeoff submission file. Its arguments are your `predict_one` function and an output filename (str).

In [91]:
def create_bakeoff_submission(
        predict_one_func,
        output_filename='cs224u-sentiment-bakeoff-entry.csv'):

    bakeoff_test = sst.bakeoff_test_reader(SST_HOME)
    sst_test = sst.test_reader(SST_HOME)
    bakeoff_test['dataset'] = 'bakeoff'
    sst_test['dataset'] = 'sst3'
    df = pd.concat((bakeoff_test, sst_test))

    df['prediction'] = df['sentence'].apply(predict_one_func)

    df.to_csv(output_filename, index=None)
    
    

Thus, for example, the following will create a bake-off entry based on `predict_one_softmax`:

In [92]:
# This check ensure that the following code only runs on the local environment only.
# The following call will not be run on the autograder environment.
if 'IS_GRADESCOPE_ENV' not in os.environ:
    create_bakeoff_submission(predict_one_rnn)
    
    

D:\Anaconda\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


This creates a file `cs224u-sentiment-bakeoff-entry.csv` in the current directory. That file should be uploaded as-is. Please do not change its name.

Only one upload per team is permitted, and you should do no tuning of your system based on what you see in our bakeoff prediction file – you should not study that file in anyway, beyond perhaps checking that it contains what you expected it to contain. The upload function will do some additional checking to ensure that your file is well-formed.

People who enter will receive the additional homework point, and people whose systems achieve the top score will receive an additional 0.5 points. We will test the top-performing systems ourselves, and only systems for which we can reproduce the reported results will win the extra 0.5 points.

Late entries will be accepted, but they cannot earn the extra 0.5 points.

## Submission instructions

Review and follow the [Homework and bake-off code: Formatting guide](hw_formatting_guide.ipynb).
Please do not change the filename as described below.

Submit the following files to Gradescope:

- `hw_sentiment.ipynb` (this notebook)
- `cs224u-sentiment-bakeoff-entry.csv` (bake-off output)
